# Notebook 10 — Conclusions

This notebook states what the project found and contributes. Nb 08 made the case; nb 10 declares it.

## 1. What we did

We formulated Q-learning as Bayesian linear regression on Monte-Carlo return targets, with a hierarchical Normal prior shared across the three per-action coefficient blocks and a near-Jeffreys Inverse-Gamma prior on the residual variance. We applied the model to daily SPY ETF data from 1994 through 2026, with a 2020-04 onward test window. We compared per-episode posterior-sampling Thompson action selection against ε-greedy classical Q-learning baselines — tabular Q on quantile-binned features and linear Fitted-Q-Iteration — with a uniform-random policy as a sanity floor. Robustness was tested across asset (TLT), likelihood (Student-t scale mixture), cost regime (0.5×/2×), and prior strength (`nu_sigma ∈ {0.1, 1.0, 4.0}`). A validity check confirmed that posterior credible intervals are invariant to the σ²-prior choice across three orders of magnitude in effective prior sample size.

## 2. What we found

**Bayesian and classical Q-learning are statistically indistinguishable on mean performance.** On the SPY test window the Bayesian (Gaussian) policy returns mean Sharpe `-1.520` with 95% credible interval `[-2.580, -0.313]`; this interval overlaps the linear-FQI seed distribution on every headline metric (Sharpe, max drawdown, turnover) — and continues to overlap on TLT, under the Student-t likelihood, and under the 0.5× and 2× cost multipliers (nb 05, nb 06). Where mean estimates differ, the differences sit inside ensemble uncertainty.

**The calibration is data-dominated and prior-invariant.** The nb 09 Panel A sensitivity score — the maximum mean shift across the prior sweep divided by the median 95% interval width — is `0.000` to printed precision on every metric, across `nu_sigma ∈ {0.1, 1.0, 4.0}`. Roughly 6,500 training rows dominate at most four prior pseudo-observations by three orders of magnitude. The 500-path Monte-Carlo budget for posterior-predictive intervals is also converged (Panel C scores below `0.05`).

**Three qualifications attach to the headline.** First, all four methods produce negative mean Sharpe on the 2020-04 to 2026 test window: this regime — Covid crash, 2022 bear, 2023–24 rally — defeats the linear-feature, three-action policy class regardless of which estimator produces it. Second, the "Bayesian beats tabular" reading from nb 05 is conditional on `n_bins=4`: at `n_bins=2` the tabular Sharpe collapses to within `0.06` of the Bayesian mean, and nb 09 Panel B flags every metric QUALIFIED (scores `1.6`–`3.7`). Third, per-episode Thompson is empirically dominant (nb 07): per-step Thompson collapses to Sharpe `-4.61` with 95% CrI `[-5.31, -3.95]`, non-overlapping with per-episode and trading 3.6× the turnover.

## 3. What this contributes

**Methodological.** A fully Bayesian formulation of Q-learning with each piece principled and tested: priors elicited from the empirical residual standard deviation $\hat\sigma_y$ rather than asserted, hierarchical pooling across actions through a shared $\mu_\beta$, conjugate Gibbs blocks for the Gaussian variant and a partial-collapse Liu–Wong–Kong scheme for the Student-t variant, per-episode posterior-sampling Thompson selection (nb 07), and an empirical validity check that turns the calibration claim from rhetoric into evidence (nb 09 Panel A). The contribution is the framework — explicit prior, likelihood, posterior update, cadence, validated calibration — rather than the empirical result it produces.

**Empirical.** A *calibrated null*. On a single liquid asset with ample training data, Bayesian and well-tuned classical Q-learning produce statistically indistinguishable mean performance, *and* the Bayesian credible intervals are demonstrably data-dominated rather than prior-driven. This characterizes a regime in which the methods agree, with quantified uncertainty on both sides — a positive empirical validation of Bernstein–von Mises convergence in the regular parametric Q-regression setting, not a failed treatment effect.

## 4. Next steps

- **Multi-asset hierarchical extension.** This project pools across the three actions; the natural follow-up shares information across assets through a between-asset variance term — the regime where the Bayesian-RL literature most credibly locates an empirical advantage.
- **Asymmetric-loss decision rules.** A drawdown-averse trader weights the lower tail more heavily than the upper; the posterior over $Q(s, a)$ is strictly more useful than a point estimate under non-symmetric loss, and that machinery was deliberately not exercised by the symmetric Sharpe / MDD / turnover headline metrics.
- **Bayesian function approximation beyond linearity.** Gaussian-process Q-learning or sparse-prior models on richer feature sets address the linearity qualifier nb 08 §4 names; the conjugate skeleton generalises.
- **TD-bootstrap-streaming Q-learning comparison.** The variant deferred from the headline (Decision #5): MC-target supervised regression and TD-bootstrap converge on the same $Q$ in the limit, but their finite-sample equivalence on real financial data is untested under the project's robustness sweeps.